# Experiment 0: Pipeline Sanity

This notebook runs the end-to-end Experiment 0 pipeline.

In [ ]:
!pip -q install alpaca-py pandas numpy pyarrow scikit-learn


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_ROOT = Path('/content/drive/MyDrive/ETF Duel Foundation Model Project/codex_build')
RAW_DIR = OUTPUT_ROOT / 'data' / 'raw'
PROCESSED_DIR = OUTPUT_ROOT / 'data' / 'processed'
RUNS_DIR = OUTPUT_ROOT / 'runs'
MODELS_DIR = OUTPUT_ROOT / 'models'

for path in [RAW_DIR, PROCESSED_DIR, RUNS_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd()
if (repo_root / 'src').exists():
    sys.path.insert(0, str(repo_root / 'src'))

from datetime import datetime, timedelta, timezone

from alpaca.data.historical import StockHistoricalDataClient
from google.colab import userdata

from etf_duel.config import coerce_symbols
from etf_duel.data_download import download_universe
from etf_duel.labels import LabelConfig
from etf_duel.pipeline import load_and_build_dataset
from etf_duel.sanity import sanity_summary, summary_to_dict

symbols = coerce_symbols(['SPY', 'QQQ', 'IWM', 'AAPL', 'MSFT'])
end = datetime.now(timezone.utc)
start = end - timedelta(days=30 * 18)

key = userdata.get('PAPER_KEY')
sec = userdata.get('PAPER_SEC')

client = StockHistoricalDataClient(api_key=key, secret_key=sec)

download_universe(
    client=client,
    symbols=symbols,
    start=start,
    end=end,
    timeframe_minutes=30,
    adjustment='split',
    output_dir=RAW_DIR,
)

label_config = LabelConfig(atr_window=14, k_up=2.0, k_dn=1.0, horizon_bars=26)
dataset = load_and_build_dataset(
    raw_dir=RAW_DIR,
    label_config=label_config,
    vol_window=20,
    volume_z_window=50,
)

dataset_path = PROCESSED_DIR / 'experiment0_dataset.parquet'
dataset.to_parquet(dataset_path, index=False)

summary = sanity_summary(dataset)
summary_path = RUNS_DIR / 'experiment0_summary.json'
summary_path.write_text(json.dumps(summary_to_dict(summary), indent=2))

print('Rows:', summary.rows)
print('Date range:', summary.start, '->', summary.end)
print('Label distribution:', summary.label_distribution)
